# Open-Data Card-Triggered Passing Conservatism

This notebook studies whether players become more conservative passers after receiving a card across the full StatsBomb open-data event set.

Current scope:
- Data source: `data/open-data/data`
- Scope: all teams in the event corpus
- Main focus: yellow cards
- Unit of comparison: same player, same match, fixed time window before vs after the card

Outputs:
- Card sample summary
- Before/after pass summary table
- Directional pass mix comparison
- Pass length comparison
- End-location heatmaps and a normalized difference heatmap

Notes:
- The main analysis still focuses on yellow cards because that sample is much larger than red cards.
- This is an exploratory notebook rather than a causal design.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)


In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root containing README.md and data/.')


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / 'data' / 'open-data' / 'data'
EVENTS_DIR = DATA_ROOT / 'events'

TEAM_SCOPE = 'all teams'
WINDOW_MINUTES = 15
WINDOW_SECONDS = WINDOW_MINUTES * 60
FORWARD_THRESHOLD = 5.0
LATERAL_BAND = 5.0
HEATMAP_BINS_X = 24
HEATMAP_BINS_Y = 16

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'DATA_ROOT    = {DATA_ROOT}')
print(f'EVENTS_DIR   = {EVENTS_DIR}')
print(f'TEAM_SCOPE   = {TEAM_SCOPE}')
print(f'WINDOW       = +/- {WINDOW_MINUTES} minutes')


In [ ]:
def read_json(path: Path) -> Any:
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def safe_name(value: Any) -> str | None:
    if value is None:
        return None
    return str(value)


def event_seconds(event: dict[str, Any]) -> int:
    minute = int(event.get('minute') or 0)
    second = int(event.get('second') or 0)
    return minute * 60 + second


def parse_xy(value: Any) -> tuple[float, float]:
    if isinstance(value, (list, tuple)) and len(value) >= 2:
        return float(value[0]), float(value[1])
    return np.nan, np.nan


def get_card_name(event: dict[str, Any]) -> str | None:
    event_type = ((event.get('type') or {}).get('name'))
    if event_type == 'Foul Committed':
        foul = event.get('foul_committed') or {}
        return ((foul.get('card') or {}).get('name'))
    if event_type == 'Bad Behaviour':
        bad = event.get('bad_behaviour') or {}
        return ((bad.get('card') or {}).get('name'))
    return None


def pass_direction(dx: float) -> str:
    if dx > FORWARD_THRESHOLD:
        return 'forward'
    if dx < -FORWARD_THRESHOLD:
        return 'backward'
    return 'lateral'


def draw_pitch(ax: plt.Axes) -> None:
    ax.set_xlim(0, 120)
    ax.set_ylim(0, 80)
    ax.set_aspect('equal')
    ax.add_patch(plt.Rectangle((0, 0), 120, 80, fill=False, edgecolor='black', lw=2))
    ax.axvline(60, color='black', lw=1)
    center_circle = plt.Circle((60, 40), 10, fill=False, color='black', lw=1)
    ax.add_patch(center_circle)
    ax.add_patch(plt.Rectangle((0, 18), 18, 44, fill=False, edgecolor='black', lw=1))
    ax.add_patch(plt.Rectangle((102, 18), 18, 44, fill=False, edgecolor='black', lw=1))
    ax.add_patch(plt.Rectangle((0, 30), 6, 20, fill=False, edgecolor='black', lw=1))
    ax.add_patch(plt.Rectangle((114, 30), 6, 20, fill=False, edgecolor='black', lw=1))
    ax.scatter([12, 108], [40, 40], color='black', s=10)
    ax.set_xticks([])
    ax.set_yticks([])


def normalized_heatmap(df: pd.DataFrame, x_col: str, y_col: str, bins_x: int = HEATMAP_BINS_X, bins_y: int = HEATMAP_BINS_Y):
    if df.empty:
        heat = np.zeros((bins_y, bins_x), dtype=float)
    else:
        heat, xedges, yedges = np.histogram2d(
            df[x_col].to_numpy(),
            df[y_col].to_numpy(),
            bins=[bins_x, bins_y],
            range=[[0, 120], [0, 80]],
        )
        heat = heat.T
        total = heat.sum()
        if total > 0:
            heat = heat / total
    return heat


In [ ]:
card_rows: list[dict[str, Any]] = []
pass_rows: list[dict[str, Any]] = []

for event_path in sorted(EVENTS_DIR.glob('*.json')):
    match_id = int(event_path.stem)
    events = read_json(event_path)

    for event in events:
        team_name = safe_name((event.get('team') or {}).get('name'))
        player_name = safe_name((event.get('player') or {}).get('name'))
        event_type = safe_name((event.get('type') or {}).get('name'))
        sec = event_seconds(event)

        card_name = get_card_name(event)
        if card_name is not None:
            card_rows.append({
                'match_id': match_id,
                'event_id': safe_name(event.get('id')),
                'index': event.get('index'),
                'period': event.get('period'),
                'minute': event.get('minute'),
                'second': event.get('second'),
                'event_seconds': sec,
                'team_name': team_name,
                'player_name': player_name,
                'card_name': card_name,
                'event_type': event_type,
            })

        if event_type == 'Pass':
            start_x, start_y = parse_xy(event.get('location'))
            pass_data = event.get('pass') or {}
            end_x, end_y = parse_xy(pass_data.get('end_location'))
            dx = end_x - start_x
            dy = end_y - start_y
            length = float(np.hypot(dx, dy)) if np.isfinite(dx) and np.isfinite(dy) else np.nan
            outcome = (pass_data.get('outcome') or {}).get('name')
            recipient = ((event.get('pass') or {}).get('recipient') or {}).get('name')

            pass_rows.append({
                'match_id': match_id,
                'event_id': safe_name(event.get('id')),
                'index': event.get('index'),
                'period': event.get('period'),
                'minute': event.get('minute'),
                'second': event.get('second'),
                'event_seconds': sec,
                'team_name': team_name,
                'player_name': player_name,
                'recipient_name': safe_name(recipient),
                'start_x': start_x,
                'start_y': start_y,
                'end_x': end_x,
                'end_y': end_y,
                'dx': dx,
                'dy': dy,
                'length': length,
                'successful': outcome is None,
                'pass_outcome_name': safe_name(outcome),
                'direction_bucket': pass_direction(dx) if np.isfinite(dx) else None,
                'progressive_pass': bool(np.isfinite(dx) and dx >= 10.0),
            })

cards = pd.DataFrame(card_rows)
passes = pd.DataFrame(pass_rows)

print(f'Loaded {len(cards):,} card events across all teams.')
print(f'Loaded {len(passes):,} passes across all teams.')
cards.head()


## Coordinate sanity check

Before interpreting `dx > 0` as a forward pass, we run a quick sanity check on shot locations.

In this open-data export, shot `x` coordinates are typically concentrated near the attacking goal for both teams, even within the same match and period. That suggests the event coordinates are already aligned to attacking direction in a way that makes `dx = end_x - start_x` a usable forward/backward proxy here.

Because of that, this notebook does **not** apply an additional left-right flip.

In [ ]:
shot_rows: list[dict[str, Any]] = []

for event_path in sorted(EVENTS_DIR.glob('*.json')):
    match_id = int(event_path.stem)
    events = read_json(event_path)
    for event in events:
        if safe_name((event.get('type') or {}).get('name')) != 'Shot':
            continue
        x, y = parse_xy(event.get('location'))
        if not np.isfinite(x):
            continue
        shot_rows.append({
            'match_id': match_id,
            'team_name': safe_name((event.get('team') or {}).get('name')),
            'period': event.get('period'),
            'shot_x': x,
            'shot_y': y,
        })

shots = pd.DataFrame(shot_rows)
shot_direction_check = (
    shots.groupby(['match_id', 'team_name', 'period'])['shot_x']
    .agg(['count', 'mean', 'median'])
    .reset_index()
)

print(f'Match-team-period shot groups: {len(shot_direction_check):,}')
print(f"Share of groups with median shot_x > 60: {(shot_direction_check['median'] > 60).mean():.3f}")
display(shot_direction_check.head(10))


In [ ]:
yellow_cards = cards[cards['card_name'] == 'Yellow Card'].copy()
red_like_cards = cards[cards['card_name'].isin(['Red Card', 'Second Yellow'])].copy()

card_counts = cards['card_name'].value_counts(dropna=False).to_dict()
print('Card counts:', card_counts)
print(f'Yellow-card events kept for main analysis: {len(yellow_cards):,}')
print(f'Red/second-yellow events available for descriptive use: {len(red_like_cards):,}')
print(f"Unique players with at least one yellow card: {yellow_cards['player_name'].nunique():,}")


In [ ]:
pass_groups = {
    key: group.sort_values('event_seconds').reset_index(drop=True)
    for key, group in passes.groupby(['match_id', 'player_name'], sort=False)
}

analysis_frames: list[pd.DataFrame] = []

for card in yellow_cards.itertuples(index=False):
    key = (card.match_id, card.player_name)
    player_passes = pass_groups.get(key)

    if player_passes is None or player_passes.empty:
        continue

    secs = player_passes['event_seconds']
    before_mask = (secs < card.event_seconds) & (secs >= card.event_seconds - WINDOW_SECONDS)
    after_mask = (secs > card.event_seconds) & (secs <= card.event_seconds + WINDOW_SECONDS)

    if not before_mask.any() or not after_mask.any():
        continue

    before = player_passes.loc[before_mask].assign(
        window='before',
        card_event_id=card.event_id,
        card_index=card.index,
        card_minute=card.minute,
        card_second=card.second,
        card_seconds=card.event_seconds,
        card_player_name=card.player_name,
    )

    after = player_passes.loc[after_mask].assign(
        window='after',
        card_event_id=card.event_id,
        card_index=card.index,
        card_minute=card.minute,
        card_second=card.second,
        card_seconds=card.event_seconds,
        card_player_name=card.player_name,
    )

    analysis_frames.extend([before, after])

analysis_df = pd.concat(analysis_frames, ignore_index=True) if analysis_frames else pd.DataFrame()

valid_card_ids = (
    analysis_df.groupby(['card_event_id', 'window'])['event_id']
    .count()
    .unstack(fill_value=0)
)
valid_card_ids = valid_card_ids[(valid_card_ids.get('before', 0) > 0) & (valid_card_ids.get('after', 0) > 0)].index
analysis_df = analysis_df[analysis_df['card_event_id'].isin(valid_card_ids)].copy()

print(f"Yellow cards with at least one pass both before and after: {analysis_df['card_event_id'].nunique():,}")
print(f'Passes in analysis table: {len(analysis_df):,}')
analysis_df.head()


In [ ]:
summary = (
    analysis_df.groupby('window')
    .agg(
        passes=('event_id', 'count'),
        unique_cards=('card_event_id', 'nunique'),
        unique_players=('player_name', 'nunique'),
        success_rate=('successful', 'mean'),
        mean_length=('length', 'mean'),
        median_length=('length', 'median'),
        mean_dx=('dx', 'mean'),
        forward_share=('direction_bucket', lambda s: (s == 'forward').mean()),
        lateral_share=('direction_bucket', lambda s: (s == 'lateral').mean()),
        backward_share=('direction_bucket', lambda s: (s == 'backward').mean()),
        progressive_share=('progressive_pass', 'mean'),
    )
    .sort_index()
)

display(summary.style.format({
    'success_rate': '{:.3f}',
    'mean_length': '{:.2f}',
    'median_length': '{:.2f}',
    'mean_dx': '{:.2f}',
    'forward_share': '{:.3f}',
    'lateral_share': '{:.3f}',
    'backward_share': '{:.3f}',
    'progressive_share': '{:.3f}',
}))

delta = (summary.loc['after'] - summary.loc['before']).to_frame('after_minus_before')
display(delta.style.format('{:.3f}'))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

direction_plot = (
    analysis_df.groupby(['window', 'direction_bucket'])['event_id']
    .count()
    .rename('count')
    .reset_index()
)
direction_plot['share'] = direction_plot.groupby('window')['count'].transform(lambda s: s / s.sum())

sns.barplot(data=direction_plot, x='direction_bucket', y='share', hue='window', ax=axes[0])
axes[0].set_title('Directional Pass Mix')
axes[0].set_xlabel('Direction bucket')
axes[0].set_ylabel('Share of passes')

sns.kdeplot(data=analysis_df, x='length', hue='window', fill=True, common_norm=False, ax=axes[1])
axes[1].set_title('Pass Length Distribution')
axes[1].set_xlabel('Pass length')
axes[1].set_ylabel('Density')

plt.tight_layout()
plt.show()


In [ ]:
before_df = analysis_df[analysis_df['window'] == 'before'].copy()
after_df = analysis_df[analysis_df['window'] == 'after'].copy()

before_heat = normalized_heatmap(before_df, 'end_x', 'end_y')
after_heat = normalized_heatmap(after_df, 'end_x', 'end_y')
diff_heat = after_heat - before_heat

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, heat, title, cmap, vmin, vmax in [
    (axes[0], before_heat, 'Before card: normalized end-location heatmap', 'YlOrRd', 0, None),
    (axes[1], after_heat, 'After card: normalized end-location heatmap', 'YlOrRd', 0, None),
    (axes[2], diff_heat, 'After minus before', 'coolwarm', -np.max(np.abs(diff_heat)), np.max(np.abs(diff_heat))),
]:
    draw_pitch(ax)
    im = ax.imshow(
        heat,
        extent=[0, 120, 0, 80],
        origin='lower',
        cmap=cmap,
        alpha=0.85,
        aspect='auto',
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

plt.tight_layout()
plt.show()


In [ ]:
print('Player-level tables were intentionally removed to keep the notebook concise.')


## Suggested next steps

If the pooled before/after patterns look meaningful, the next upgrades are:

1. Control for game state, especially whether the booked player's team was leading, tied, or trailing at the moment of the card.
2. Exclude cards late in the match or cards followed quickly by substitution.
3. Split the analysis by player role so defenders and midfielders are not mixed together.
4. Compare pass-start locations as well as pass-end locations.
5. Connect this notebook to the pass-selection or pass-success model so conservatism can be measured relative to available options, not only raw pass outcomes.